## Properties
 - Curvature
 - Congruence
 - Cartilage thickness
 - Volume

In [13]:
import os
import pyvista as pv
import numpy as np
import pandas as pd
from pathlib import Path
os.listdir('../../Computational/notebooks/MeshPipeline/ParamOptimisation/cartilageHeightMetrics/outputs/subs_okJustified')

['meshes', 'params', 'reports']

## Cartilage thickness & Volume

In [46]:
def compute_cart_h(mesh2d, min_h):
    """"return cartilage inner region height and proportion of inner area below min height"""

    bone_shell = mesh2d.extract_cells(mesh2d['region_id']==2, invert=True).extract_surface(algorithm=None)
    inner_cart = mesh2d.extract_cells(mesh2d['inner_cells']==1).extract_surface(algorithm=None)
    cart_height = inner_cart.cell_centers().compute_implicit_distance(bone_shell)['implicit_distance']

    As = inner_cart.compute_cell_sizes()['Area']
    A_inner = np.sum(As)

    below_mask = cart_height < min_h # cell centers below min height
    A_below = np.sum(As[below_mask]) / A_inner # proportion of area below min height

    return cart_height, A_below

n_tets, min_size = 2.5, 0.06
min_h = n_tets * min_size

data2d = []
mesh2d_root = Path('../../Computational/notebooks/MeshPipeline/ParamOptimisation/cartilageHeightMetrics/outputs/subs_okJustified')
for mesh_path in mesh2d_root.glob('**/tpm-mc1/2Dmesh/bone_cartilage_mesh*.vtp'):
    sub = mesh_path.parent.parent.parent.name
    bone, _ = mesh_path.parent.parent.name.split('-')
    mesh2d = pv.read(mesh_path)
    cart_h, A_below = compute_cart_h(mesh2d, min_h)

    data2d.append({
    'sub': sub,
    #'bone': bone,
    'vol': mesh2d.extract_cells(mesh2d['region_id']!=2).extract_surface(algorithm=None).volume,
    'h_mean': cart_h.mean(),
    'h_min': cart_h.min(),
    'h_max': cart_h.max(),
    'h_std': np.std(cart_h),
    'A_below %': A_below*100,
    })

In [47]:
data2d = pd.DataFrame(data2d)
data2d.sort_values('h_mean')

,sub,vol,h_mean,h_min,h_max,h_std,A_below %
12,50021R,1962.800909,0.264730,0.065893,0.587196,0.120868,24.458144
34,14819R,1992.133810,0.266091,0.024729,0.651162,0.137645,24.227123
1,14874R,1431.566389,0.273480,0.039107,0.629595,0.142659,28.060641
21,50029R,1639.698985,0.299387,0.015210,0.609025,0.125528,14.487151
35,15283R,2334.097725,0.329545,0.093736,0.606070,0.101766,2.701862
19,50045R,2682.245092,0.340526,0.154803,0.675983,0.118881,0.000000
11,15294R,1974.350058,0.345202,0.022595,0.672916,0.141316,9.382295
27,50014R,1850.027231,0.357814,0.168335,0.730456,0.122832,0.000000
10,14726R,3486.783434,0.358475,0.140836,0.777017,0.121613,0.773333
3,50000R,1306.219341,0.369456,0.175002,0.568304,0.077913,0.000000


## Curvature & Congruence

In [30]:
SurfaceFit_df = pd.read_csv('/Users/maro/Library/CloudStorage/OneDrive-UniversityofLeeds/PhD-wrist/KineticsData/PreTransfer/TMC-SurfaceFit/TMC-SurfaceFit9.csv')
SurfaceFit_df['sub'] = SurfaceFit_df['subject'].astype(str) + SurfaceFit_df['side']
SurfaceFit_df = SurfaceFit_df[['sub', 'tpm_kmin', 'tpm_kmax', 'C_avg']]
SurfaceFit_df


,sub,tpm_kmin,tpm_kmax,C_avg
0,14548R,-0.067296,0.161986,0.645208
1,14613R,-0.058498,0.119960,0.680178
2,14685R,-0.072100,0.178106,0.635856
3,14726R,-0.075875,0.130703,0.662128
4,14727R,-0.051320,0.170361,0.637574
5,14818R,-0.048495,0.169315,0.594971
6,14819R,-0.076729,0.140710,0.683497
7,14827L,-0.063573,0.159568,0.637482
8,14873R,-0.071738,0.161735,0.632656
9,14874R,-0.042214,0.147120,0.615702


In [39]:
df = pd.merge(data2d[['sub', 'vol', 'h_mean']], SurfaceFit_df[['sub', 'tpm_kmax', 'C_avg']], 'left')
df

,sub,vol,h_mean,tpm_kmax,C_avg
0,14818R,1746.384695,0.450350,0.169315,0.594971
1,14874R,1431.566389,0.273480,0.147120,0.615702
2,22306R,2035.846440,0.377446,0.170077,0.611883
3,50000R,1306.219341,0.369456,0.156357,0.606298
4,50049R,2204.010193,0.410095,0.175239,0.752482
5,50034R,2637.685507,0.500352,0.129075,0.696403
6,15006R,1908.122916,0.413284,0.167935,0.580231
7,50017L,1755.695333,0.399737,0.163874,0.658208
8,15441R,1676.514722,0.384603,0.125899,0.704667
9,14827L,1384.925597,0.376764,0.159568,0.637482


## Rank them based on expected contact area range

In [ ]:

ranks = df.copy()

# rank each column
ranks["rank_vol"]      = ranks["vol"].rank(method="min", ascending=True)      # lower is smaller
ranks["rank_h_mean"]   = ranks["h_mean"].rank(method="min", ascending=True)   # lower is smaller
ranks["rank_tpm_kmax"] = ranks["tpm_kmax"].rank(method="min", ascending=False) # higher is smaller
ranks["rank_C_avg"]    = ranks["C_avg"].rank(method="min", ascending=True)    # lower is smaller

# sort subs by your priority order
ranks = ranks.sort_values(
    by=["rank_h_mean", "rank_tpm_kmax", "rank_C_avg", "rank_vol"]
).reset_index(drop=True)

ranks["mean_rank"] = (
    ranks["rank_h_mean"]
    + ranks["rank_tpm_kmax"]
    + ranks["rank_C_avg"]
    + ranks["rank_vol"]
) / 4

ranks = ranks.sort_values(
    by=["mean_rank", "rank_h_mean", "rank_tpm_kmax", "rank_C_avg", "rank_vol"]
).reset_index(drop=True)

ranks

,sub,vol,h_mean,tpm_kmax,C_avg,rank_vol,rank_h_mean,rank_tpm_kmax,rank_C_avg,mean_rank
0,14874R,1431.566389,0.273480,0.147120,0.615702,3.0,3.0,25.0,5.0,9.00
1,50000R,1306.219341,0.369456,0.156357,0.606298,1.0,10.0,22.0,3.0,9.00
2,15006R,1908.122916,0.413284,0.167935,0.580231,15.0,19.0,8.0,1.0,10.75
3,22306R,2035.846440,0.377446,0.170077,0.611883,21.0,13.0,6.0,4.0,11.00
4,14818R,1746.384695,0.450350,0.169315,0.594971,9.0,26.0,7.0,2.0,11.00
5,14827L,1384.925597,0.376764,0.159568,0.637482,2.0,11.0,19.0,15.0,11.75
6,50019R,1776.014120,0.383919,0.164319,0.623364,11.0,14.0,15.0,7.0,11.75
7,50029R,1639.698985,0.299387,0.164655,0.664235,7.0,4.0,13.0,26.0,12.50
8,50045R,2682.245092,0.340526,0.181367,0.634904,32.0,6.0,1.0,11.0,12.50
9,50007L,1859.471106,0.469678,0.174350,0.631871,13.0,29.0,4.0,9.0,13.75


In [ ]:
'50000R',  # Small contact area ~ 230803 cells
'50017L',  # Middle             ~ 272410 cells
'50034R',  # High contact area  ~ 293287 cells

In [ ]:
Figure out what cell count can run on uni laptop 
Then try and use aire

In [ ]:
x = df.copy()

x["r_vol"] = x["vol"].rank(pct=True)
x["r_h_mean"] = x["h_mean"].rank(pct=True)
x["r_tpm_kmax"] = 1 - x["tpm_kmax"].rank(pct=True)  # reverse
x["r_C_avg"] = x["C_avg"].rank(pct=True)

rank_cols = ["r_vol", "r_h_mean", "r_tpm_kmax", "r_C_avg"]

# closest to 0.5 on every metric
x["middle_score"] = ((x[rank_cols] - 0.5) ** 2).sum(axis=1) ** 0.5

result = x.sort_values("middle_score")
result

,sub,vol,h_mean,tpm_kmax,C_avg,r_vol,r_h_mean,r_tpm_kmax,r_C_avg,middle_score
7,50017L,1755.695333,0.399737,0.163874,0.658208,0.277778,0.472222,0.416667,0.638889,0.276385
31,50024R,1981.158812,0.417891,0.164512,0.631396,0.500000,0.611111,0.361111,0.222222,0.329843
13,50027L,1871.258940,0.460828,0.146185,0.662752,0.388889,0.750000,0.694444,0.694444,0.387896
28,50019R,1776.014120,0.383919,0.164319,0.623364,0.305556,0.388889,0.388889,0.194444,0.394796
23,50018L,2179.553758,0.376963,0.155192,0.691126,0.611111,0.333333,0.611111,0.833333,0.404451
27,50014R,1850.027231,0.357814,0.143548,0.644551,0.333333,0.222222,0.750000,0.500000,0.409192
15,14548R,1435.135271,0.447427,0.161986,0.645208,0.111111,0.694444,0.444444,0.555556,0.441833
22,50020R,2356.033275,0.438103,0.167029,0.681659,0.750000,0.666667,0.277778,0.777778,0.465640
29,50053R,2481.650726,0.498627,0.158147,0.645092,0.777778,0.888889,0.555556,0.527778,0.481926
9,14827L,1384.925597,0.376764,0.159568,0.637482,0.055556,0.305556,0.500000,0.416667,0.492223
